# Engagement 2 · Mobile Money Agent Network — Audit, Cleaning & Summary

In [1]:
import pandas as pd, numpy as np
from cleaning_pipeline import load_raw, audit_data, clean_agent_data
pd.options.display.float_format = "{:,.2f}".format
txns, agents = load_raw()   # read as text: 'N/A' is preserved, only empty cells are NaN
print("transactions:", txns.shape, "| agents:", agents.shape)
txns.head()

transactions: (612, 5) | agents: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


## Task 10 · Data-quality audit (raw extract)

In [2]:
audit = audit_data(txns, agents)
audit

,issue,dimension,column,rows_affected
0,Fully duplicated rows (same txn_id and all fie...,Uniqueness,all columns,11
1,Repeated txn_id (includes duplicates whose age...,Uniqueness,txn_id,12
2,txn_type inconsistent case / trailing whitespace,Consistency,txn_type,384
3,"txn_type outside {Deposit, Withdrawal} after n...",Validity,txn_type,0
4,amount missing (empty cell),Completeness,amount,15
5,amount is text placeholder ('N/A'),Validity,amount,7
6,"amount stored as text, not numeric",Validity,amount,597
7,amount is IQR outlier (> Q3+1.5*IQR),Accuracy,amount,40
8,"amount equals exactly 250,000 (repeated suspic...",Accuracy,amount,5
9,txn_date uses '/' instead of ISO '-' separator,Consistency,txn_date,5


Note: pandas' default reader would turn `"N/A"` into NaN and hide 7 invalid strings inside the 22 missing amounts. Reading with `keep_default_na=False, na_values=[""]` separates the 15 empty cells from the 7 `N/A` strings; the pipeline treats both as missing.

## Task 11 · Uniqueness

In [3]:
dups = txns[txns["txn_id"].duplicated(keep=False)].sort_values("txn_id")
print("rows with a repeated txn_id:", len(dups), "-> extra copies:", txns["txn_id"].duplicated().sum())
print("fully identical duplicate rows:", txns.duplicated().sum())
dups.head(6)

rows with a repeated txn_id: 24 -> extra copies: 12
fully identical duplicate rows: 11


,txn_id,agent_id,txn_type,amount,txn_date
23,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
605,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
66,TX00067,AG002,deposit,N/A,2026-04-07
608,TX00067,AG002,deposit,N/A,2026-04-07
165,TX00166,AG010,deposit,4146.0,2026-04-18
609,TX00166,AG010,deposit,4146.0,2026-04-18


12 extra copies share a `txn_id` with an earlier row, but only 11 are byte-identical: in one pair the later `AG099` injection changed `agent_id` on a single copy. Because `txn_id` is the business key, the rule is *drop repeated `txn_id`, keep first* → **12 rows removed (612 → 600)**.

## Tasks 12–16 · Run the cleaning pipeline

In [4]:
clean, log = clean_agent_data(txns, agents)
log

,step,rule,rows_affected,rows_after
0,0 Load,Raw rows received,0,612
1,1 Uniqueness,Drop repeated txn_id (keep first),12,600
2,2 Consistency,Standardise txn_type to Deposit/Withdrawal (ro...,376,600
3,3a Validity,Convert amount to numeric (text placeholders -...,20,600
4,3b Validity,Parse txn_date; '/' separators converted,5,600
5,5 Accuracy,"Flag IQR outliers (bounds -1,928 to 6,084)",40,600
6,4 Completeness,Impute missing/invalid amount with per-type me...,20,600
7,6 Integrity,"Left-join agent register; unknown agents kept,...",7,600


In [5]:
print(clean["txn_type"].value_counts(dropna=False).to_dict())
print(clean.dtypes[["amount","txn_date"]])
print("txn_date range:", clean.txn_date.min().date(), "to", clean.txn_date.max().date())
print("dup txn_id after cleaning:", clean.txn_id.duplicated().sum())

{'Deposit': 373, 'Withdrawal': 227}
amount             float64
txn_date    datetime64[us]
dtype: object
txn_date range: 2026-03-01 to 2026-04-29
dup txn_id after cleaning: 0


### Task 14 · Completeness decision
**20 amounts are unusable** after de-duplication (15 empty + 5 `N/A`; the raw file shows 22 because 2 `N/A` rows were duplicate copies). Options: delete, leave as NaN, or impute.
- *Deleting* removes real transactions (the agent did handle a deposit/withdrawal) and understates counts.
- *Leaving NaN* is the most honest for value, but makes totals silently lower.
- **Chosen: impute the median amount of the same `txn_type`, computed from valid non-outlier amounts, and flag every imputed row with `amount_imputed = True`.** The median is used because amounts are right-skewed (a mean would be dragged up by outliers). At 3.3% of rows the effect is small, and the flag lets anyone re-run any figure excluding imputed values (sensitivity check below). Imputed values are placeholders for analysis, **not** financial facts, and must never be used for reconciliation.

In [6]:
print("imputed rows:", clean.amount_imputed.sum(), f"({clean.amount_imputed.mean():.1%})")
a = clean.groupby("county")["amount"].sum()
b = clean[~clean.amount_imputed].groupby("county")["amount"].sum()
pd.DataFrame({"incl_imputed": a, "excl_imputed": b, "diff_%": (a/b-1)*100})

imputed rows: 20 (3.3%)


,incl_imputed,excl_imputed,diff_%
county,,,
Kiambu,"411,657.00","410,011.00",0.40
Kisumu,"512,486.00","505,689.00",1.34
Mombasa,"517,340.00","510,472.00",1.35
Nairobi,"704,866.00","691,414.00",1.95
Nakuru,"513,235.00","508,155.00",1.00
Unknown,"27,156.00","27,156.00",0.00


### Task 15 · Outliers (IQR)

In [7]:
obs = clean.loc[~clean.amount_imputed, "amount"]
q1,q3 = obs.quantile([.25,.75]); iqr=q3-q1
print(f"Q1={q1:,.0f}  Q3={q3:,.0f}  IQR={iqr:,.0f}  upper fence={q3+1.5*iqr:,.0f}")
print("flagged is_outlier:", clean.is_outlier.sum())
# robustness: IQR on log scale (amounts are right-skewed)
lg=np.log(obs); lq1,lq3=lg.quantile([.25,.75]); lf=np.exp(lq3+1.5*(lq3-lq1))
print(f"log-scale upper fence = {lf:,.0f}  -> flagged: {(obs>lf).sum()}")
big = clean[clean.amount==250000]
print("\nExactly KES 250,000:", len(big), "| max of all other amounts:", f"{clean.loc[clean.amount!=250000,'amount'].max():,.0f}")
big[["txn_id","agent_id","txn_type","amount","txn_date","county"]]

Q1=1,077  Q3=3,080  IQR=2,003  upper fence=6,084
flagged is_outlier: 40
log-scale upper fence = 14,895  -> flagged: 8

Exactly KES 250,000: 5 | max of all other amounts: 28,847


,txn_id,agent_id,txn_type,amount,txn_date,county
306,TX00307,AG012,Deposit,"250,000.00",2026-03-02,Kisumu
315,TX00316,AG014,Deposit,"250,000.00",2026-04-01,Nairobi
424,TX00425,AG005,Withdrawal,"250,000.00",2026-03-02,Mombasa
463,TX00464,AG009,Withdrawal,"250,000.00",2026-04-11,Nakuru
594,TX00595,AG016,Deposit,"250,000.00",2026-03-11,Kiambu


**Finding.** On raw amounts the IQR rule flags 40 rows; most are legitimate high-value tail items because amounts are log-normal (the log-scale fence flags far fewer). The five **KES 250,000** rows are different: they are identical to the shilling, more than 100× the median and far above every other amount (the next-largest amount is KES 28,847). Identical round values suggest a system default, a cap, a test entry or keying error rather than organic activity.

**Recommendation.** Do **not** delete or silently cap them. (1) Keep and flag (`is_outlier`); (2) exclude them from averages and headline totals until verified, reporting them in a separate line; (3) refer the 5 txn_ids to agent operations / the legacy-system owner for confirmation against source records and float reconciliation; (4) if one is confirmed genuine, unflag it, otherwise correct at source; (5) add a validation rule/limit alert at ingestion.

## Task 16 · Merge validation

In [8]:
print("raw rows:", len(txns), "| after de-dup:", len(clean), "| rows after merge:", len(clean))
print("transactions with unknown agents:", clean.unknown_agent.sum())
print("unknown agent_ids:", clean.loc[clean.unknown_agent,"agent_id"].unique())
print("value on unknown-agent rows: KES", f"{clean.loc[clean.unknown_agent,'amount'].sum():,.0f}")
assert len(clean)==clean.txn_id.nunique()
clean[clean.unknown_agent].head()

raw rows: 612 | after de-dup: 600 | rows after merge: 600
transactions with unknown agents: 7
unknown agent_ids: <StringArray>
['AG099']
Length: 1, dtype: str
value on unknown-agent rows: KES 27,156


,txn_id,agent_id,txn_type,amount,txn_date,is_outlier,amount_imputed,county,onboarded,unknown_agent,week_start
55,TX00056,AG099,Deposit,"8,507.00",2026-04-01,True,False,Unknown,NaN,True,2026-03-30
68,TX00069,AG099,Withdrawal,"1,877.00",2026-04-24,False,False,Unknown,NaN,True,2026-04-20
77,TX00078,AG099,Deposit,"1,770.00",2026-04-06,False,False,Unknown,NaN,True,2026-04-06
237,TX00238,AG099,Deposit,"4,944.00",2026-04-06,False,False,Unknown,NaN,True,2026-04-06
267,TX00268,AG099,Deposit,"3,110.00",2026-04-09,False,False,Unknown,NaN,True,2026-04-06


`AG099` is not in the 20-agent register. The 8 injected rows fall to **7 after de-duplication** (one injected row was a duplicate copy that was dropped). They are **kept** (they are real money movements) with `county = 'Unknown'` and `unknown_agent = True`, so they appear in a separate line of every summary and are escalated to the data owner to confirm whether AG099 is an unregistered/decommissioned agent or a keying error. A left-join was used so no rows were lost; `validate='m:1'` guards against register duplicates.

## Task 17 · Summaries

In [9]:
outl = clean[~clean.is_outlier]   # headline figures exclude flagged outliers; see note below
def summ(d):
    s = d.groupby(["county","txn_type"])["amount"].agg(total_value="sum", n_txns="count", avg_value="mean").round(2)
    return s
print("(a) County summary — ALL transactions")
county_summary = summ(clean); display(county_summary)
print("(a2) County summary — EXCLUDING IQR outliers")
display(summ(outl))

(a) County summary — ALL transactions


total_value  n_txns  avg_value
county  txn_type                                  
Kiambu  Deposit      334,737.00      43   7,784.58
        Withdrawal    76,920.00      27   2,848.89
Kisumu  Deposit      436,242.00      74   5,895.16
        Withdrawal    76,244.00      37   2,060.65
Mombasa Deposit      148,748.00      72   2,065.94
        Withdrawal   368,592.00      61   6,042.49
Nairobi Deposit      543,352.00     118   4,604.68
        Withdrawal   161,514.00      62   2,605.06
Nakuru  Deposit      184,771.00      60   3,079.52
        Withdrawal   328,464.00      39   8,422.15
Unknown Deposit       25,279.00       6   4,213.17
        Withdrawal     1,877.00       1   1,877.00

(a2) County summary — EXCLUDING IQR outliers


total_value  n_txns  avg_value
county  txn_type                                  
Kiambu  Deposit       84,737.00      42   2,017.55
        Withdrawal    48,199.00      23   2,095.61
Kisumu  Deposit      128,823.00      67   1,922.73
        Withdrawal    59,814.00      35   1,708.97
Mombasa Deposit      124,003.00      69   1,797.14
        Withdrawal   110,630.00      59   1,875.08
Nairobi Deposit      236,841.00     110   2,153.10
        Withdrawal   120,062.00      57   2,106.35
Nakuru  Deposit      118,824.00      55   2,160.44
        Withdrawal    70,999.00      37   1,918.89
Unknown Deposit       16,772.00       5   3,354.40
        Withdrawal     1,877.00       1   1,877.00

In [10]:
print("(b) Weekly transaction value by county (week starting Monday), all transactions")
weekly = clean.pivot_table(index="week_start", columns="county", values="amount", aggfunc="sum", fill_value=0, margins=True, margins_name="Total").round(0)
weekly

(b) Weekly transaction value by county (week starting Monday), all transactions


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_start,,,,,,,
2026-02-23 00:00:00,0.00,"1,893.00","6,759.00","7,046.00","2,684.00",0.00,"18,382.00"
2026-03-02 00:00:00,"18,317.00","287,933.00","285,610.00","68,967.00","56,574.00",0.00,"717,401.00"
2026-03-09 00:00:00,"267,641.00","23,186.00","30,418.00","52,801.00","28,730.00",0.00,"402,776.00"
2026-03-16 00:00:00,"22,054.00","27,935.00","25,085.00","43,048.00","39,795.00","1,639.00","159,556.00"
2026-03-23 00:00:00,"11,504.00","27,077.00","17,050.00","41,314.00","18,868.00",0.00,"115,813.00"
2026-03-30 00:00:00,"22,609.00","41,077.00","38,344.00","282,931.00","18,145.00","8,507.00","411,613.00"
2026-04-06 00:00:00,"15,030.00","15,341.00","47,033.00","56,727.00","276,609.00","9,824.00","420,564.00"
2026-04-13 00:00:00,"12,265.00","29,297.00","26,730.00","56,302.00","26,093.00",0.00,"150,687.00"
2026-04-20 00:00:00,"24,507.00","37,439.00","16,687.00","64,926.00","30,862.00","1,877.00","176,298.00"


In [11]:
clean_out = clean.copy()
clean_out["txn_date"] = clean_out["txn_date"].dt.strftime("%Y-%m-%d")
clean_out["week_start"] = clean_out["week_start"].dt.strftime("%Y-%m-%d")
clean_out.to_csv("outputs/agent_transactions_clean.csv", index=False)
chk = pd.read_csv("outputs/agent_transactions_clean.csv")
print(chk.shape, chk.txn_id.is_unique, sorted(chk.txn_type.unique()), list(chk.columns))

(600, 11) True ['Deposit', 'Withdrawal'] ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'is_outlier', 'amount_imputed', 'county', 'onboarded', 'unknown_agent', 'week_start']


**Reading the summaries.** The five KES 250,000 rows distort the picture: Kiambu deposits fall from KES 334,737 to 84,737 once outliers are excluded, and Mombasa/Nakuru withdrawals show the same effect. With outliers excluded, **Nairobi** is clearly the busiest county (167 transactions, ~KES 357k), and deposits outnumber withdrawals in every county. The weekly pivot's spikes (weeks of 2 Mar, 9 Mar, 30 Mar, 6 Apr) are single 250,000 entries, not real demand surges. Headline reporting should therefore use the outlier-excluded table until the 5 transactions are verified.

## Task 19 + Data Quality Report

# Data Quality Report — Agent Transaction Extract
**Scope:** `agent_transactions.csv` (612 rows) merged with `agents.csv` (20 agents). **Result:** 600 unique transactions, 7 on unknown agent AG099.

| Dimension | Finding | Action | Rows |
|---|---|---|---|
| **Completeness** | `amount` missing: 15 empty cells + 7 `N/A` text placeholders in the raw file (20 unusable after de-dup) | Median imputation by `txn_type`, flagged `amount_imputed` | 20 |
| **Uniqueness** | 12 repeated `txn_id` (11 exact copies + 1 copy whose `agent_id` was altered) | Drop repeats, keep first | 12 removed |
| **Validity** | `amount` stored as text; `N/A` strings; no unparseable dates | Converted to numeric; invalid → missing → imputed | 597 converted |
| **Consistency** | `txn_type` in 5 spellings (case, trailing space); `txn_date` with `/` for 5 rows | Mapped to `Deposit`/`Withdrawal`; dates normalised to ISO `YYYY-MM-DD` | 376 / 5 |
| **Accuracy** | 5 identical KES 250,000 amounts, more than 100× the median; 40 rows beyond the IQR fence overall | Flagged `is_outlier`, kept, excluded from headline averages; refer for source verification | 40 (5 critical) |
| **Integrity (referential)** | `agent_id = AG099` absent from the register | Kept, `county='Unknown'`, `unknown_agent` flag; escalate | 7 |

**Limitations.** Imputed amounts are analytical placeholders only. County totals include outliers unless stated. Only 60 days (Mar–Apr 2026) are covered.

### Privacy — Kenya Data Protection Act 2019
If the extract also held customer phone numbers, **the phone number (MSISDN) would be personal data** under the Act, and combined with `txn_date`, `amount`, `agent_id` and location it becomes a financial-behaviour profile that can re-identify a person. Names, national ID and any device/location identifiers would also count; the Act's 'sensitive' list does not name financial data, but it is high-risk and should be handled as such. Protection: (1) **data minimisation** (s.25) — do not ingest the phone number unless needed for the analysis; (2) **pseudonymise** it with a salted hash or tokenised customer ID, keeping the key separate; (3) **restrict access** (role-based) and encrypt at rest and in transit; (4) define a lawful basis and purpose limitation, a retention period and deletion rule; (5) analysts work on the pseudonymised copy and publish only aggregates (suppress small counts); (6) register with the Office of the Data Protection Commissioner as a controller/processor, run a DPIA (s.31), keep a breach-notification procedure (72 hours, s.43) and respect data-subject rights; (7) restrict cross-border transfers (s.48–50) unless safeguards exist.